In [227]:
import pandas as pd

df = pd.read_csv("../Project Root/data/raw/retail_store_sales.csv")

df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,4/8/2024,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,7/23/2023,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,10/5/2022,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,5/7/2022,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,10/2/2022,False


In [228]:
df.shape

(12575, 11)

In [229]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Discount Applied'],
      dtype='object')

In [230]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [231]:
df.sample(5, random_state = 1)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
1804,TXN_7450962,CUST_19,Butchers,Item_12_BUT,21.5,10.0,215.0,Cash,Online,1/30/2022,NaN
4485,TXN_3415090,CUST_04,Electric household essentials,NaN,35.0,NaN,NaN,Digital Wallet,In-store,1/20/2023,False
8944,TXN_7304590,CUST_19,Electric household essentials,Item_2_EHE,6.5,10.0,65.0,Cash,In-store,5/14/2022,False
854,TXN_7557655,CUST_11,Furniture,Item_20_FUR,33.5,8.0,268.0,Credit Card,Online,4/4/2022,NaN
2745,TXN_3377748,CUST_16,Electric household essentials,Item_8_EHE,15.5,4.0,62.0,Cash,In-store,7/2/2023,NaN


## Initial data profiling
- Rows: 12575
- Columns:11
- Key observations: of the five df samples, three had NaN values.
-
-

## Column classification (Initial)
|  Column Name    |   Type   |   Description  |   Likely Table  |
|-----------------/----------/----------------/-----------------/
 Transaction ID   | object   | 12575 non-null |    Transaction 
 Customer ID      | object   | 12575 non-null |    Customer 
 Category         | object   | 12575 non-null |    Store 
 Item             | object   | 11362 non-null |    Product 
 Price Per Unit   | float64  | 11966 non-null |    Product
 Quantity         | float64  | 11971 non-null |    Product
 Total Spent      | float64  | 11971 non-null |    Transaction
 Payment Method   | object   | 12575 non-null |    Transaction 
 Location         | object   | 12575 non-null |    Store 
 Transaction Date | object   | 12575 non-null |    Date/Time 
 Discount Applied | object   | 8376 non-null  |    Transaction 

In [232]:
df.isna().sum().sort_values(ascending=False)

Discount Applied    4199
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Transaction ID         0
Customer ID            0
Category               0
Payment Method         0
Location               0
Transaction Date       0
dtype: int64

- Column with the highest missing values: <"Discount_Applied">(4199 of rows)
- Column with no missing values <Transaction ID, Customer ID, Category, Payment Method, Location, Transaction Date>

In [233]:
(df.isna().mean() *100).sort_values(ascending=False)

Discount Applied    33.391650
Item                 9.646123
Price Per Unit       4.842942
Quantity             4.803181
Total Spent          4.803181
Transaction ID       0.000000
Customer ID          0.000000
Category             0.000000
Payment Method       0.000000
Location             0.000000
Transaction Date     0.000000
dtype: float64

## Column Roles and Intended Tables
|  Column Name    |   Roles   |   Notes  |
|-----------------/----------/----------------/
 Transaction ID   |    PK    | No missing values | 
 Customer ID      |Identifier(future Foreign Key)| No missing values  |
 Category         | Attribute| No missing values | 
 Item             | Attribute| 9.65% missing  | 
 Price Per Unit   |  Metric  | 4.84% missing |
 Quantity         |  Metric  | 4.80% missing|
 Total Spent      |  Metric  | 4.80% missing |
 Payment Method   | Attribute| No missing values | 
 Location         | Attribute| No missing values | 
 Transaction Date | Date Attribute| No missing values | 
 Discount Applied |  Metric  | 33.39% missing  | 

In [234]:
missing_summary = (
    df.isna()
    .sum()
    .to_frame(name="missing_count")
    .assign(missing_percent=lambda x: (x["missing_count"]/ len(df)) * 100)
    .sort_values("missing_count", ascending = False)
)

missing_summary

,missing_count,missing_percent
Discount Applied,4199,33.391650
Item,1213,9.646123
Price Per Unit,609,4.842942
Quantity,604,4.803181
Total Spent,604,4.803181
Transaction ID,0,0.000000
Customer ID,0,0.000000
Category,0,0.000000
Payment Method,0,0.000000
Location,0,0.000000


COLUMN BY COLUMN INITIAL HYPOTHESIS

Discount Applied : MNAR
Item: MAR
Price per Unit: MAR
Quantity: MAR
Total Spent: MAR

In [235]:
df["Discount Applied"].describe()

count     8376
unique       2
top       True
freq      4219
Name: Discount Applied, dtype: object

In [236]:
df[["Item", "Price Per Unit", "Quantity", "Total Spent"]].isna().corr()

,Item,Price Per Unit,Quantity,Total Spent
Item,1.000000,0.690448,0.687464,0.687464
Price Per Unit,0.690448,1.000000,-0.050674,-0.050674
Quantity,0.687464,-0.050674,1.000000,1.000000
Total Spent,0.687464,-0.050674,1.000000,1.000000


## Missing Data Classification

- Discount Applied: MNAR (likely not recorded when no discount applied)
- Item: MAR (missingness may depend on category or ingestion issues)
- Price Per Unit: MAR (often missing alongside item)
- Quantity: MAR (transaction-level incompleteness)
- Total Spent: Derived metric; missing when upstream values are missing

## Missing Value Treatment Decisions

- Discount Applied: Impute with False (MNAR, no discount applied)
- Item: Leave NULL to avoid creating categorical data.
- Price Per Unit: Leave NULL until validated
- Quantity: Leave NULL until validated
- Total Spent: Derived from price x quantity - discount
- Other Tables: Leave as-is.

In [237]:
df_clean = df.copy()

In [238]:
df_clean["Discount Applied"] = df_clean["Discount Applied"].fillna(False)

C:\Users\pro\AppData\Local\Temp\ipykernel_24412\2887470288.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean["Discount Applied"] = df_clean["Discount Applied"].fillna(False)


In [239]:
df_clean["Discount Applied"].value_counts(dropna = False)

Discount Applied
False    8356
True     4219
Name: count, dtype: int64

In [240]:
df_clean["Discount Applied"].isna().sum()

np.int64(0)

In [241]:
df_clean["Discount Applied"].describe()

count     12575
unique        2
top       False
freq       8356
Name: Discount Applied, dtype: object

In [242]:
df_clean["Discount Applied"].dtype

dtype('bool')

## Documentation Update

- Discount Applied is a boolean flag indicating whether a promotion was used.
- Missing values were imputed as False, representing no discount.

In [243]:
df_clean["Expected Total"] = (
    df_clean["Price Per Unit"] * df_clean["Quantity"]
)

In [244]:
df_clean["Total Difference"] = (
    df_clean["Total Spent"] - df_clean["Expected Total"]
)

In [245]:
df_clean["Total Difference"].describe()

count    11362.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: Total Difference, dtype: float64

In [246]:
tolerance = 0.01

mismatches = df_clean[
    df_clean["Total Difference"].abs() > tolerance
    ]

mismatches.shape

(0, 13)

In [247]:
mismatches.sample(5, random_state=42)

ValueError: a must be greater than 0 unless no samples are taken

In [248]:
df_clean["Total Difference"].abs().max()

0.0

In [249]:
df_clean.drop(
    columns=["Expected Total", "Total Difference"],
    inplace=True
)

In [250]:
numeric_cols = [
    "Price Per Unit",
    "Quantity",
    "Total Spent"
]

In [251]:
df_clean[numeric_cols].describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [252]:
outlier_summary = {}

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper_bound_total_spent = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower) | (df_clean[col] > 
                                   
                                   upper_bound_total_spent)
        ]
    
    outlier_summary[col] = outliers.shape[0]

In [253]:
outlier_summary


{'Price Per Unit': 0, 'Quantity': 0, 'Total Spent': 60}

In [254]:
df_clean["High_Value_Transaction"] = (
    df_clean["Total Spent"] > upper_bound_total_spent
)

In [255]:
def flag_high_value_transactions(df):
    Q1 = df["Total Spent"].quantile(0.25)
    Q3 = df["Total Spent"].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    return df["Total Spent"] > upper

df_clean["High_Value_Transaction"] = flag_high_value_transactions(df_clean)

In [256]:
df_clean["Transaction Date"] = pd.to_datetime(df_clean["Transaction Date"])

In [257]:
df_clean["Transaction Date"].dtype

dtype('<M8[ns]')

## Create Time Based Attributes.

In [258]:
df_clean["Year"] = df_clean["Transaction Date"].dt.year
df_clean["Month"] = df_clean["Transaction Date"].dt.month
df_clean["Month_Name"] = df_clean["Transaction Date"].dt.month_name()
df_clean["Day"] = df_clean["Transaction Date"].dt.day
df_clean["Weekday"] = df_clean["Transaction Date"].dt.day_name()

## Business Metrics

In [259]:
df_clean["Revenue"] = df_clean["Price Per Unit"] * df_clean["Quantity"]

In [260]:
df_clean["Discount_Flag"] = df_clean["Discount Applied"].astype(int)

In [261]:
df_clean["High_Value_Transaction"] = (
    df_clean["Total Spent"] > upper_bound_total_spent
)

In [262]:
text_cols = ["Category", "Item", "Payment Method", "Location"]

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip().str.title()

In [263]:
df_clean.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,High_Value_Transaction,Year,Month,Month_Name,Day,Weekday,Revenue,Discount_Flag
0,TXN_6867343,CUST_09,Patisserie,Item_10_Pat,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,False,2024,4,April,8,Monday,185.0,1
1,TXN_3731986,CUST_22,Milk Products,Item_17_Milk,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,False,2023,7,July,23,Sunday,261.0,1
2,TXN_9303719,CUST_02,Butchers,Item_12_But,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,False,2022,10,October,5,Wednesday,43.0,0
3,TXN_9458126,CUST_06,Beverages,Item_16_Bev,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,False,2022,5,May,7,Saturday,247.5,0
4,TXN_4575373,CUST_05,Food,Item_6_Food,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,False,2022,10,October,2,Sunday,87.5,0


In [264]:
dim_product = (
    df_clean[["Category", "Item"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product["Product_ID"] = dim_product.index + 1
dim_product

,Category,Item,Product_ID
0,Patisserie,Item_10_Pat,1
1,Milk Products,Item_17_Milk,2
2,Butchers,Item_12_But,3
3,Beverages,Item_16_Bev,4
4,Food,Item_6_Food,5
...,...,...,...
203,Food,Item_16_Food,204
204,Food,Item_19_Food,205
205,Electric Household Essentials,Item_14_Ehe,206
206,Electric Household Essentials,Item_3_Ehe,207


In [265]:
dim_location = (
    df_clean[["Location"]]
    .drop_duplicates()
    .reset_index(drop = True)
)

dim_location["Location_ID"] = dim_location.index + 1
dim_location

,Location,Location_ID
0,Online,1
1,In-Store,2


In [266]:
dim_customer = (
    df_clean[["Customer ID"]]
    .drop_duplicates()
    .reset_index(drop = True)
)

dim_customer

,Customer ID
0,CUST_09
1,CUST_22
2,CUST_02
3,CUST_06
4,CUST_05
5,CUST_07
6,CUST_21
7,CUST_23
8,CUST_25
9,CUST_14


In [267]:
dim_date = (
    df_clean[["Transaction Date"]]
    .drop_duplicates()
    .sort_values("Transaction Date")
    .reset_index(drop = True)
)

dim_date["Year"] = dim_date["Transaction Date"].dt.year
dim_date["Month"] = dim_date["Transaction Date"].dt.month
dim_date["Month_Name"] = dim_date["Transaction Date"].dt.month_name()
dim_date["Day"] = dim_date["Transaction Date"].dt.day
dim_date["Day_Of_Week"] = dim_date["Transaction Date"].dt.day_name()

dim_date

,Transaction Date,Year,Month,Month_Name,Day,Day_Of_Week
0,2022-01-01,2022,1,January,1,Saturday
1,2022-01-02,2022,1,January,2,Sunday
2,2022-01-03,2022,1,January,3,Monday
3,2022-01-04,2022,1,January,4,Tuesday
4,2022-01-05,2022,1,January,5,Wednesday
...,...,...,...,...,...,...
1109,2025-01-14,2025,1,January,14,Tuesday
1110,2025-01-15,2025,1,January,15,Wednesday
1111,2025-01-16,2025,1,January,16,Thursday
1112,2025-01-17,2025,1,January,17,Friday


In [268]:
fact_sales = df_clean.merge(
    dim_product,
    on = ["Category", "Item"],
    how = "left"
)

In [269]:
fact_sales = fact_sales.merge(
    dim_location,
    on = "Location",
    how = "left"
)

In [270]:
fact_sales = fact_sales[[
    "Transaction ID",
    "Customer ID",
    "Product_ID",
    "Location_ID",
    "Transaction Date",
    "Price Per Unit",
    "Quantity",
    "Total Spent",
    "Discount Applied",
    "High_Value_Transaction"
]]

fact_sales.head()

,Transaction ID,Customer ID,Product_ID,Location_ID,Transaction Date,Price Per Unit,Quantity,Total Spent,Discount Applied,High_Value_Transaction
0,TXN_6867343,CUST_09,1,1,2024-04-08,18.5,10.0,185.0,True,False
1,TXN_3731986,CUST_22,2,1,2023-07-23,29.0,9.0,261.0,True,False
2,TXN_9303719,CUST_02,3,1,2022-10-05,21.5,2.0,43.0,False,False
3,TXN_9458126,CUST_06,4,1,2022-05-07,27.5,9.0,247.5,False,False
4,TXN_4575373,CUST_05,5,1,2022-10-02,12.5,7.0,87.5,False,False


In [271]:
fact_sales.isna().sum()

Transaction ID              0
Customer ID                 0
Product_ID                  0
Location_ID                 0
Transaction Date            0
Price Per Unit            609
Quantity                  604
Total Spent               604
Discount Applied            0
High_Value_Transaction      0
dtype: int64

In [272]:
fact_sales.shape

(12575, 10)

In [273]:
fact_sales["Transaction ID"].is_unique

True

## Export code

In [280]:
fact_sales.to_csv("C:/Users/pro/Project Root/data/processed/fact_sales.csv", index = False)
dim_product.to_csv("C:/Users/pro/Project Root/data/processed/dim_product.csv", index = False)
dim_location.to_csv("C:/Users/pro/Project Root/data/processed/dim_location.csv", index = False)
dim_customer.to_csv("C:/Users/pro/Project Root/data/processed/dim_customer.csv", index = False)
dim_date.to_csv("C:/Users/pro/Project Root/data/processed/dim_date.csv", index = False)

In [281]:
import os
os.getcwd()

'C:\\Users\\pro\\notebook'

In [282]:
os.listdir()

['.ipynb_checkpoints']

In [283]:
os.listdir("data")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data'